# Visual Jev — Colab test harness

Runs [Visual-Jev](https://github.com/guanxuyu-sv/Visual-Jev) (Qwen3-VL-4B + answer-SFT LoRA adapter) against test frames or video, using Colab's GPU.

The model is loaded **once** into a persistent `model` variable; a reusable `ask(image=None, questions_spec=None, shared_context="", *, video=None)` function then runs against as many frames or videos as you want without ever reloading. Steps: clone repo → install deps → load model once → sanity-check on the bundled demo image → loop over your own frames (uploaded stills, or sampled from an uploaded video) → or hand an uploaded video straight to the model as native video input.

**Before running:** set Runtime → Change runtime type → T4 GPU (or better).

In [2]:
!nvidia-smi

Tue Oct  6 11:21:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# !git clone --depth 1 https://github.com/guanxuyu-sv/Visual-Jev.git
# %cd Visual-Jev

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install dependencies

Colab ships a recent CUDA-enabled torch already. Only reinstall torch (next cell, commented out) if the check below prints `cuda available: False`.

In [3]:
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

torch 2.11.0+cu130 | cuda available: True


In [4]:
# Only run this if the check above printed cuda available: False
# !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [5]:
!pip install -q -r /content/drive/MyDrive/jev_dev/code/requirements.txt opencv-python-headless

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.


## Load the model once

This downloads the Qwen3-VL-4B backbone + the `visual-jev-4b-answer-sft` LoRA adapter and keeps both in memory in `model`. Every frame afterwards reuses this same loaded model — no reload per frame.

In [1]:
!pip install -U numpy torchao

In [2]:
!pip list | grep torchao

torchao                               0.18.0


In [3]:
import os
import sys

sys.path.insert(0, os.path.abspath("/content/drive/MyDrive/jev_dev/code"))

import torch
from peft import PeftModel
from PIL import Image

from vdm.models.vdm_model import VDM

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float16 if device == "mps" else torch.bfloat16

model = VDM("Qwen/Qwen3-VL-4B-Instruct", device=device, dtype=dtype, with_heads=False)
model.processor.image_processor.max_pixels = 200704
model.backbone = PeftModel.from_pretrained(model.backbone, "guanxuyu/visual-jev-4b-answer-sft").eval()
print(f"model loaded once on {device} ({dtype})")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  132MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

model loaded once on cuda (torch.bfloat16)


## Reusable inference helper

`ask(image=None, questions_spec=None, shared_context="", *, video=None)` takes either an image (path or PIL image) or a video (path or URL), plus a dict of named `choice` questions, and returns each question's predicted option plus the full probability distribution. Exactly one of `image`/`video` is required. It never touches the model-loading code — just `prepare_group` + a forward pass — so calling it in a loop is cheap relative to the one-time load above.

In [4]:
def _build_question(instructions, criteria):
    option_ids, candidates = [], []
    for option_id, description in criteria.items():
        option_ids.append(option_id)
        candidates.append(option_id if not description else f"{option_id}: {description}")
    return {"qtype": "choice", "instruction": instructions, "candidates": candidates}, option_ids


def ask(image=None, questions_spec=None, shared_context="", *, video=None):
    """questions_spec: {question_id: {"instructions": str, "criteria": {option: description_or_None}}}
    Returns {question_id: {"prediction": str, "probabilities": {option: float}}}.
    Exactly one of `image` (a path or PIL image) or `video` (a path or URL) must be given.
    Reuses the already-loaded `model` global -- no reload per call."""
    if isinstance(image, str):
        with Image.open(image) as source:
            image = source.convert("RGB")

    qids = list(questions_spec.keys())
    questions, option_id_lists = [], []
    for qid in qids:
        spec = questions_spec[qid]
        q, option_ids = _build_question(spec["instructions"], spec["criteria"])
        questions.append(q)
        option_id_lists.append(option_ids)

    with torch.inference_mode():
        group = (
            model.prepare_group(image=image, questions=questions, shared_context=shared_context)
            if image is not None
            else model.prepare_group(video=video, questions=questions, shared_context=shared_context)
        )
        run = model.run_independent if len(questions) == 1 else model.run_prefix_share_batch
        output = run(group)

    results = {}
    for i, qid in enumerate(qids):
        logits = output["lm_option_logits"][i, : group.n_options[i]]
        probs = torch.softmax(logits.float(), dim=-1).cpu().tolist()
        option_ids = option_id_lists[i]
        prediction = option_ids[max(range(len(probs)), key=probs.__getitem__)]
        results[qid] = {"prediction": prediction, "probabilities": dict(zip(option_ids, probs))}
    return results

## Sanity check: bundled demo image, six questions, one shared-prefix batch

Reproduces the README's example without spawning a subprocess or reloading the model.

In [ ]:
import json

with open("code/examples/bottle_questions.json") as f:
    bottle_request = json.load(f)

bottle_results = ask(
    "assets/figures/demo_bottle.jpg",
    bottle_request["questions"],
    shared_context=bottle_request["state"],
)
for qid, r in bottle_results.items():
    print(f"{qid}: {r['prediction']}  {r['probabilities']}")

## Define the driving question set once

Reused for every frame below.

In [5]:
driving_questions = {
    "action": {
        "instructions": "What should the car do next?",
        "criteria": {"accelerate": None, "brake": None, "change_lane": None, "maintain_speed": None},
    },
    "obstacle_present": {
        "instructions": "Is there an obstacle (vehicle, pedestrian, or object) directly ahead in this lane?",
        "criteria": {"yes": None, "no": None},
    },
}

## Loop over uploaded still frames

Upload one or more images. Each is passed through the already-loaded `model` in turn — no reload between frames.

In [ ]:
from google.colab import files

uploaded = files.upload()
frame_paths = list(uploaded.keys())
print("uploaded:", frame_paths)

In [8]:
import time
frame_paths = ["/content/drive/MyDrive/jev_dev/driving_cam.mp4"]
frame_results = []
for path in frame_paths:
    t0 = time.time()
    r = ask(questions_spec=driving_questions, video=path)
    elapsed = time.time() - t0
    print(f"{path}  ({elapsed:.2f}s)")
    for qid, answer in r.items():
        print(f"  {qid}: {answer['prediction']}  {answer['probabilities']}")
    frame_results.append({"frame": path, **r})

[transformers] Qwen3VL video processing does not apply the per-frame pixel cap the reference implementation (qwen-vl-utils) applies, so some videos cost far more tokens than they would there. In v5.22 the capped behavior will become the default and `cap_pixels_per_frame` will be removed. Pass `cap_pixels_per_frame=True` to adopt the reference behavior now, or `False` to keep the current behavior and silence this warning.


/content/drive/MyDrive/jev_dev/driving_cam.mp4  (122.02s)
  action: brake  {'accelerate': 0.0213128924369812, 'brake': 0.9062472581863403, 'change_lane': 0.0213128924369812, 'maintain_speed': 0.05112697556614876}
  obstacle_present: no  {'yes': 0.43782350420951843, 'no': 0.562176525592804}


## Loop over frames sampled from an uploaded video

For a *per-frame* decision (e.g. "what should the car do right now, at this instant"): sample every `sample_every_n_frames`-th frame and feed each one through the same persistent `model` as an independent image, one decision per sampled frame.

For a single decision over the *whole* clip instead, see the native-video section below, which passes the uploaded video straight to `ask(video=...)`.

In [ ]:
video_uploaded = files.upload()
video_path = next(iter(video_uploaded))
print("uploaded video:", video_path)

In [ ]:
import cv2

sample_every_n_frames = 15  # e.g. ~2 fps from a 30fps source

cap = cv2.VideoCapture(video_path)
video_results = []
frame_idx = 0
while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break
    if frame_idx % sample_every_n_frames == 0:
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)
        r = ask(pil_image, driving_questions)
        print(f"frame {frame_idx}: action={r['action']['prediction']}  obstacle={r['obstacle_present']['prediction']}")
        video_results.append({"frame_idx": frame_idx, **r})
    frame_idx += 1
cap.release()
print(f"processed {len(video_results)} sampled frames out of {frame_idx} total")

## Native video input: one decision over the whole clip

Visual Jev can take a video directly as the shared visual context, the same way it takes an image — one video, multiple questions, one shared-prefix batch. This reuses the video already uploaded above, with no manual frame sampling.

In [ ]:
video_results_native = ask(video=video_path, questions_spec=driving_questions)
for qid, r in video_results_native.items():
    print(f"{qid}: {r['prediction']}  {r['probabilities']}")